# BPE Function-by-Function Debug Notebook

Run cells top-to-bottom. Each section tests one function from `cs336_basics.bpe` with fixed inputs.

In [20]:
from pathlib import Path
from tempfile import TemporaryDirectory
from collections import Counter
import importlib
import sys

# Ensure project root is importable regardless of notebook working directory
cwd = Path.cwd().resolve()
project_root = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "cs336_basics").exists():
        project_root = candidate
        break
if project_root is None:
    raise RuntimeError("Could not locate project root containing cs336_basics and pyproject.toml")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import cs336_basics.bpe as bpe
importlib.reload(bpe)

print("Loaded:", bpe.__name__)
print("Project root on sys.path:", project_root)
print(sys.executable)

Loaded: cs336_basics.bpe
Project root on sys.path: C:\Course\cs336\assignment1-basics
c:\Course\cs336\assignment1-basics\.venv\Scripts\python.exe


In [21]:
def check(name, condition, detail=''):
    if condition:
        print(f'PASS: {name}')
    else:
        print(f'FAIL: {name} {detail}')

## 1) find_chunk_boundaries

In [22]:
with TemporaryDirectory() as td:
    p = Path(td) / 'sample.txt'
    data = b'aa<|endoftext|>bb<|endoftext|>cc'
    p.write_bytes(data)

    boundaries = bpe.find_chunk_boundaries(
        file_path=p,
        desired_num_chunks=3,
        split_special_token=b'<|endoftext|>',
    )

    print('boundaries =', boundaries)
    check('starts at 0', boundaries[0] == 0)
    check('ends at file size', boundaries[-1] == len(data))
    check('sorted', boundaries == sorted(boundaries))
    check('unique', len(boundaries) == len(set(boundaries)))

boundaries = [0, 17, 32]
PASS: starts at 0
PASS: ends at file size
PASS: sorted
PASS: unique


## 2) bpe_vocab_init

In [23]:
vocab = bpe.bpe_vocab_init(vocab_size=300, special_tokens=['<|endoftext|>'])

check('byte 0 in vocab', vocab[0] == b'\x00')
check('byte 255 in vocab', vocab[255] == b'\xff')
check('special token id', vocab[256] == b'<|endoftext|>')

PASS: byte 0 in vocab
PASS: byte 255 in vocab
PASS: special token id


## 3) bpe_pre_token_bytes_seqs_with_counts

In [24]:
pre_tokens = Counter({b'ab': 2, b'aa': 1})
pre_bytes_seqs_with_counts, pair_counts = bpe.bpe_pre_token_bytes_seqs_with_counts(pre_tokens)

print('pre_bytes_seqs_with_counts =', pre_bytes_seqs_with_counts)
print('pair_counts =', pair_counts)

# Normalize keys so checks work whether keys are tuple[bytes, ...] or list[bytes]
seq_keys = []
for k in pre_bytes_seqs_with_counts.keys():
    seq_keys.append(list(k) if isinstance(k, tuple) else k)

check('contains [a,b]', [b'a', b'b'] in seq_keys)
check('contains [a,a]', [b'a', b'a'] in seq_keys)
check('(a,b) count == 2', pair_counts[(b'a', b'b')] == 2)
check('(a,a) count == 1', pair_counts[(b'a', b'a')] == 1)

pre_bytes_seqs_with_counts = Counter({(b'a', b'b'): 2, (b'a', b'a'): 1})
pair_counts = Counter({(b'a', b'b'): 2, (b'a', b'a'): 1})
PASS: contains [a,b]
PASS: contains [a,a]
PASS: (a,b) count == 2
PASS: (a,a) count == 1


## 4) bpe_find_max_freq

In [25]:
pair_counts = Counter({
    (b'a', b'b'): 3,
    (b'a', b'c'): 3,
    (b'b', b'a'): 2,
})

max_pair, max_count = bpe.bpe_find_max_freq(pair_counts)
print('max_pair =', max_pair, 'max_count =', max_count)

check('max count is 3', max_count == 3)
check('lexicographically greater tie-break', max_pair == (b'a', b'c'))

max_pair = (b'a', b'c') max_count = 3
PASS: max count is 3
PASS: lexicographically greater tie-break


## 5) bpe_merge

In [26]:
pre_bytes_seqs_with_counts = Counter({(b'a', b'b', b'a', b'b', b'x'): 1})
max_pair = (b'a', b'b')
counts = Counter({(b'a', b'b'): 2, (b'b', b'a'): 1, (b'b', b'x'): 1})

updated_pre_bytes_seqs_with_counts, updated_counts = bpe.bpe_merge(
    pre_bytes_seqs_with_counts,
    max_pair,
    counts,
)

print('updated_pre_bytes_seqs_with_counts =', updated_pre_bytes_seqs_with_counts)
print('updated_counts sample =', dict(list(updated_counts.items())[:5]))

# Normalize keys so checks work whether keys are tuple[bytes, ...] or list[bytes]
updated_seq_keys = []
for k in updated_pre_bytes_seqs_with_counts.keys():
    updated_seq_keys.append(list(k) if isinstance(k, tuple) else k)

check('non-overlapping left-to-right merge', [b'ab', b'ab', b'x'] in updated_seq_keys)

updated_pre_bytes_seqs_with_counts = Counter({(b'ab', b'ab', b'x'): 1})
updated_counts sample = {(b'ab', b'ab'): 1, (b'ab', b'x'): 1}
PASS: non-overlapping left-to-right merge


In [29]:
# Extra corner-case checks based on current bpe.py behavior

print('--- find_chunk_boundaries corner cases ---')
with TemporaryDirectory() as td:
    p = Path(td) / 'no_special.txt'
    data = b'abcdefg'
    p.write_bytes(data)

    boundaries = bpe.find_chunk_boundaries(
        file_path=p,
        desired_num_chunks=8,
        split_special_token=b'<|endoftext|>',
    )
    print('boundaries (no split token) =', boundaries)
    check('no split token -> only [0, file_size]', boundaries == [0, len(data)])

print('\n--- bpe_vocab_init corner case ---')
try:
    _ = bpe.bpe_vocab_init(vocab_size=256, special_tokens=['<|endoftext|>'])
    check('vocab_size too small should assert', False)
except AssertionError:
    check('vocab_size too small should assert', True)

print('\n--- bpe_pre_token_bytes_seqs_with_counts corner cases ---')
pre_tokens = Counter({b'a': 3, b'': 2})
pre_bytes_seqs_with_counts, pair_counts = bpe.bpe_pre_token_bytes_seqs_with_counts(pre_tokens)
print('pre_bytes_seqs_with_counts =', pre_bytes_seqs_with_counts)
print('pair_counts =', pair_counts)
check('single-byte token creates no pairs', (b'a', b'a') not in pair_counts)
check('empty token creates empty tuple key', () in pre_bytes_seqs_with_counts)

print('\n--- bpe_find_max_freq corner case ---')
max_pair, max_count = bpe.bpe_find_max_freq(Counter())
print('empty counter result =', max_pair, max_count)
check('empty pair counter -> (None, None)', max_pair is None and max_count is None)

print('\n--- bpe_merge corner cases ---')
# Overlapping matches: (a,a,a) with max_pair=(a,a) should merge left-to-right once
pre = Counter({(b'a', b'a', b'a'): 1})
counts = Counter({(b'a', b'a'): 2})
updated_pre, updated_counts = bpe.bpe_merge(pre, (b'a', b'a'), counts)
print('updated_pre (overlap) =', updated_pre)
check('overlap merges once left-to-right', (b'aa', b'a') in updated_pre)

# Single-element sequences should be unchanged
pre_single = Counter({(b'z',): 5})
updated_single, _ = bpe.bpe_merge(pre_single, (b'a', b'b'), Counter())
print('updated_pre (single) =', updated_single)
check('single-token sequence unchanged', updated_single == pre_single)

print('\n--- bpe_pre_tokenize guard-rail asserts ---')
with TemporaryDirectory() as td:
    p = Path(td) / 'tiny.txt'
    p.write_text('hello', encoding='utf-8')

    try:
        _ = bpe.bpe_pre_tokenize(p, special_tokens=['<|endoftext|>'], num_workers=0)
        check('num_workers <= 0 should assert', False)
    except AssertionError:
        check('num_workers <= 0 should assert', True)

    try:
        _ = bpe.bpe_pre_tokenize(p, special_tokens=[], num_workers=1)
        check('empty special_tokens should assert', False)
    except AssertionError:
        check('empty special_tokens should assert', True)

--- find_chunk_boundaries corner cases ---
boundaries (no split token) = [0, 7]
PASS: no split token -> only [0, file_size]

--- bpe_vocab_init corner case ---
PASS: vocab_size too small should assert

--- bpe_pre_token_bytes_seqs_with_counts corner cases ---
pre_bytes_seqs_with_counts = Counter({(b'a',): 3, (): 2})
pair_counts = Counter()
PASS: single-byte token creates no pairs
PASS: empty token creates empty tuple key

--- bpe_find_max_freq corner case ---
empty counter result = None None
PASS: empty pair counter -> (None, None)

--- bpe_merge corner cases ---
updated_pre (overlap) = Counter({(b'aa', b'a'): 1})
PASS: overlap merges once left-to-right
updated_pre (single) = Counter({(b'z',): 5})
PASS: single-token sequence unchanged

--- bpe_pre_tokenize guard-rail asserts ---
PASS: num_workers <= 0 should assert
PASS: empty special_tokens should assert


## 6) More Corner Cases

In [ ]:
print('--- find_chunk_boundaries edge positioning ---')
with TemporaryDirectory() as td:
    p = Path(td) / 'empty.bin'
    p.write_bytes(b'')
    boundaries = bpe.find_chunk_boundaries(
        file_path=p,
        desired_num_chunks=4,
        split_special_token=b'<|endoftext|>',
    )
    print('empty file boundaries =', boundaries)
    check('empty file boundaries collapse to [0]', boundaries == [0])

with TemporaryDirectory() as td:
    p = Path(td) / 'start_end_token.bin'
    tok = b'<|endoftext|>'
    data = tok + b'abc' + tok
    p.write_bytes(data)
    boundaries = bpe.find_chunk_boundaries(
        file_path=p,
        desired_num_chunks=3,
        split_special_token=tok,
    )
    print('start/end token boundaries =', boundaries)
    check('contains 0 boundary', 0 in boundaries)
    check('contains file end boundary', len(data) in boundaries)

print('\n--- bpe_update_vocab sequencing ---')
vocab2 = bpe.bpe_vocab_init(vocab_size=260, special_tokens=['<|endoftext|>'])
old_max = max(vocab2.keys())
bpe.bpe_update_vocab(vocab2, b'ab')
check('new token id increments max by 1', old_max + 1 in vocab2)
check('new token bytes inserted at new id', vocab2[old_max + 1] == b'ab')

print('\n--- bpe_pre_token_bytes_seqs_with_counts UTF-8 bytes ---')
utf8_token = '你好'.encode('utf-8')
pre_tokens = Counter({utf8_token: 1})
pre_bytes_seqs_with_counts, pair_counts = bpe.bpe_pre_token_bytes_seqs_with_counts(pre_tokens)
print('utf8 seq keys =', list(pre_bytes_seqs_with_counts.keys()))
print('utf8 pair counts =', pair_counts)
check('utf8 token splits into raw bytes', len(next(iter(pre_bytes_seqs_with_counts.keys()))) == len(utf8_token))
check('pair count length is n-1', sum(pair_counts.values()) == max(len(utf8_token) - 1, 0))

print('\n--- bpe_find_max_freq bytes tie-break ---')
pair_counts = Counter({(b'a', b'z'): 5, (b'b', b'a'): 5, (b'a', b'a'): 1})
max_pair, max_count = bpe.bpe_find_max_freq(pair_counts)
print('max_pair =', max_pair, 'max_count =', max_count)
check('tie-break picks lexicographically greatest pair', max_pair == (b'b', b'a'))
check('tie-break keeps max count', max_count == 5)

print('\n--- bpe_merge count update neighbors ---')
pre = Counter({(b'x', b'a', b'b', b'y'): 2})
updated_pre, updated_counts = bpe.bpe_merge(pre, (b'a', b'b'), Counter())
print('updated_pre =', updated_pre)
print('updated_counts =', updated_counts)
check('sequence merges to x,ab,y', (b'x', b'ab', b'y') in updated_pre)
check('left neighbor count updated', updated_counts[(b'x', b'ab')] == 2)
check('right neighbor count updated', updated_counts[(b'ab', b'y')] == 2)

print('\n--- bpe_merge additive across sequences ---')
pre = Counter({(b'a', b'b', b'c'): 1, (b'a', b'b', b'd'): 3})
updated_pre, updated_counts = bpe.bpe_merge(pre, (b'a', b'b'), Counter())
print('updated_pre additive =', updated_pre)
print('updated_counts additive =', updated_counts)
check('merged key [ab,c] present', (b'ab', b'c') in updated_pre)
check('merged key [ab,d] present', (b'ab', b'd') in updated_pre)
check('counts accumulate with multiplicity', updated_counts[(b'ab', b'd')] == 3)

--- find_chunk_boundaries edge positioning ---
empty file boundaries = [0]
PASS: empty file boundaries collapse to [0]
start/end token boundaries = [0, 16, 29]
PASS: contains 0 boundary
PASS: contains file end boundary

--- bpe_update_vocab sequencing ---


KeyError: b'ab'

## Optional: quick smoke for bpe_pre_tokenize

If this fails with regex errors, your pattern uses `\p{L}`/`\p{N}` which requires the third-party `regex` package (not built-in `re`).

In [ ]:
with TemporaryDirectory() as td:
    p = Path(td) / 'tiny.txt'
    p.write_text('hello world <|endoftext|> hello', encoding='utf-8')
    try:
        c = bpe.bpe_pre_tokenize(p, special_tokens=['<|endoftext|>'], num_workers=1)
        print('bpe_pre_tokenize returned', len(c), 'unique pre-tokens')
    except Exception as e:
        print('bpe_pre_tokenize error:', type(e).__name__, str(e))

Starting token counting across all chunks...
Processed chunk 1/1...
bpe_pre_tokenize returned 4 unique pre-tokens


## E2E Test

In [ ]:
with TemporaryDirectory() as td:
    p = Path(td) / 'sample.txt'
    data = b'aa<|endoftext|>bb<|endoftext|>cc abcad efg aba efg'
    p.write_bytes(data)

    vocab_size = 260
    special_tokens = ['<|endoftext|>']
    vocab = bpe.bpe_vocab_init(vocab_size, special_tokens)
    merges = []
    pre_tokens = bpe.bpe_pre_tokenize(p, special_tokens, 1)
    pre_bytes_seqs_with_counts, bytes_pair_counts = bpe.bpe_pre_token_bytes_seqs_with_counts(pre_tokens)
    print("vocab:", vocab)
    print("pre_tokens:", pre_tokens)
    print("pre_bytes_seqs_with_counts:", pre_bytes_seqs_with_counts)
    print("bytes_pair_counts:", bytes_pair_counts)
    
    for i in range(1):
        max_pair, max_count = bpe.bpe_find_max_freq(bytes_pair_counts)
        if max_pair is None:
            break
        print(f"Merge {i + 1}: {max_pair} (count: {max_count})")
        bytes_pair_counts.pop(max_pair)
        vocab[len(vocab)] = max_pair[0] + max_pair[1]
        merges.append(max_pair)
        pre_bytes_seqs_with_counts, bytes_pair_counts = bpe.bpe_merge(pre_bytes_seqs_with_counts, max_pair, bytes_pair_counts)
        print("pre_bytes_seqs_with_counts after merge:", pre_bytes_seqs_with_counts)
        print("bytes_pair_counts after merge:", bytes_pair_counts)
        print("vocab after merge:", vocab)

Starting token counting across all chunks...
Processed chunk 1/1...
vocab: {b'\x00': 0, b'\x01': 1, b'\x02': 2, b'\x03': 3, b'\x04': 4, b'\x05': 5, b'\x06': 6, b'\x07': 7, b'\x08': 8, b'\t': 9, b'\n': 10, b'\x0b': 11, b'\x0c': 12, b'\r': 13, b'\x0e': 14, b'\x0f': 15, b'\x10': 16, b'\x11': 17, b'\x12': 18, b'\x13': 19, b'\x14': 20, b'\x15': 21, b'\x16': 22, b'\x17': 23, b'\x18': 24, b'\x19': 25, b'\x1a': 26, b'\x1b': 27, b'\x1c': 28, b'\x1d': 29, b'\x1e': 30, b'\x1f': 31, b' ': 32, b'!': 33, b'"': 34, b'#': 35, b'$': 36, b'%': 37, b'&': 38, b"'": 39, b'(': 40, b')': 41, b'*': 42, b'+': 43, b',': 44, b'-': 45, b'.': 46, b'/': 47, b'0': 48, b'1': 49, b'2': 50, b'3': 51, b'4': 52, b'5': 53, b'6': 54, b'7': 55, b'8': 56, b'9': 57, b':': 58, b';': 59, b'<': 60, b'=': 61, b'>': 62, b'?': 63, b'@': 64, b'A': 65, b'B': 66, b'C': 67, b'D': 68, b'E': 69, b'F': 70, b'G': 71, b'H': 72, b'I': 73, b'J': 74, b'K': 75, b'L': 76, b'M': 77, b'N': 78, b'O': 79, b'P': 80, b'Q': 81, b'R': 82, b'S': 83, b'T'